In [1]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
ismailnasri20_driver_drowsiness_dataset_ddd_path = kagglehub.dataset_download('ismailnasri20/driver-drowsiness-dataset-ddd')

print('Data source import complete.')


Using Colab cache for faster access to the 'driver-drowsiness-dataset-ddd' dataset.
Data source import complete.


In [2]:
import os
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import load_img
from tensorflow.keras.preprocessing.image import img_to_array
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, GlobalAveragePooling2D,BatchNormalization
from tensorflow.keras.applications import InceptionResNetV2

In [3]:
base_path = "/kaggle/input/driver-drowsiness-dataset-ddd/Driver Drowsiness Dataset (DDD)"

drowsy_path = os.path.join(base_path, "Drowsy")
non_drowsy_path = os.path.join(base_path, "Non Drowsy")

In [4]:
# Data PreProcessing
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=20,
    horizontal_flip=True,
    zoom_range=0.2,
)

img_target_size = (299, 299)

# Train Data
train_gen = datagen.flow_from_directory(
    base_path,
    target_size=img_target_size,
    batch_size=32,
    class_mode="binary",
    subset="training",
    shuffle=True
)

# Test Data
val_gen = datagen.flow_from_directory(
    base_path,
    target_size=img_target_size,
    batch_size=32,
    class_mode="binary",
    subset="validation",
    shuffle=True
)

Found 33435 images belonging to 2 classes.
Found 8358 images belonging to 2 classes.


**CNN Model**

In [11]:
base_model = InceptionResNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

219055592/219055592 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [12]:
# Methods to support inception_resnet_v2 architecture
from tensorflow.keras.layers import Conv2D, BatchNormalization, Activation, Concatenate, Add, Input
from tensorflow.keras.models import Model

def conv_fn(x, filters, kernel, strides=1, bias=True):
    x = Conv2D(filters, kernel, strides=strides, padding='same', use_bias=bias)(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    return x

def maxpool_fn(x, pool_size=(3, 3), strides=1, padding='same'):
    return MaxPooling2D(pool_size=pool_size, strides=strides, padding=padding)(x)

In [13]:
# STEM Method
def stem_maker_fn(x):
    x = conv_fn(x, 32, (3,3), strides=2)
    x = conv_fn(x, 32, (3,3))
    x = conv_fn(x, 64, (3,3))

    branch_1 = maxpool_fn(x, (3, 3), strides=2)
    branch_2 = conv_fn(x, 96, (3,3), strides=2)

    x = Concatenate()([branch_1, branch_2])

    return x

In [14]:
# inception-resnet-A Method
def inception_resnet_A_maker_fn(x, scale=0.17):
    xA1 = conv_fn(x, 32, (1,1))

    xA2 = conv_fn(x, 32, (1,1))
    xA2 = conv_fn(xA2, 32, (3,3))

    xA3 = conv_fn(x, 32, (1,1))
    xA3 = conv_fn(xA3, 48, (3,3))
    xA3 = conv_fn(xA3, 64, (3,3))

    mixed = Concatenate()([xA1, xA2, xA3])
    up = Conv2D(x.shape[-1], (1,1), padding='same')(mixed)

    x = Add()([x, up * scale])
    x = Activation('relu')(x)
    return x

In [15]:
def reduction_A_maker_fn(x):
    branch1 = maxpool_fn(x, pool_size=(3, 3), strides=2, padding='valid')

    branch2 = conv_fn(x, filters=384, kernel=3, strides=2)  # valid padding handled separately
    branch2 = Conv2D(384, 3, strides=2, padding='valid', use_bias=True)(x)
    branch2 = BatchNormalization()(branch2)
    branch2 = Activation('relu')(branch2)

    branch3 = conv_fn(x, filters=256, kernel=1, strides=1)
    branch3 = conv_fn(branch3, filters=256, kernel=3, strides=1)
    branch3 = Conv2D(384, 3, strides=2, padding='valid', use_bias=True)(branch3)
    branch3 = BatchNormalization()(branch3)
    branch3 = Activation('relu')(branch3)

    x = Concatenate()([branch1, branch2, branch3])

    return x

In [16]:
def inception_resnet_B_maker_fn(x):
    # Save the input depth to ensure the residual match
    in_channels = x.shape[-1]

    branch1 = conv_fn(x, filters=192, kernel=1)

    branch2 = conv_fn(x, filters=128, kernel=1)
    branch2 = conv_fn(branch2, filters=160, kernel=(1, 7))
    branch2 = conv_fn(branch2, filters=192, kernel=(7, 1))

    mixed = Concatenate()([branch1, branch2])

    # Project to match the input depth for the Add layer
    mixed = Conv2D(in_channels, kernel_size=1, padding='same', use_bias=True)(mixed)
    mixed = BatchNormalization()(mixed)

    x = Add()([x, mixed])
    x = Activation('relu')(x)
    return x

In [17]:
def build_model(input_shape=(299,299,3)):
    inputs = Input(shape=input_shape)

    model = stem_maker_fn(inputs)

    for _ in range(5):
        model = inception_resnet_A_maker_fn(model, scale=0.17)

    model = reduction_A_maker_fn(model)

    for _ in range(5):
        model = inception_resnet_B_maker_fn(model)

    model = GlobalAveragePooling2D()(model)

    model = Dense(256, activation='relu')(model)
    model = Dropout(0.5)(model)

    model = Dense(64, activation='relu')(model)
    model = Dropout(0.3)(model)

    outputs = Dense(1, activation='sigmoid')(model)

    return Model(inputs, outputs)

In [18]:
from tensorflow.keras.optimizers import Adam
model = build_model()

model.compile(
    optimizer=Adam(learning_rate=0.0003),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 299, 299,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_203 (Conv2D) │ (None, 150, 150,  │        896 │ input_layer_1[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 150, 150,  │        128 │ conv2d_203[0][0]  │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_203      │ (None, 150, 150,  │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_204 (Conv2D) │ (None, 150, 150,  │      9,248 │ activation_203[0… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 150, 150,  │        128 │ conv2d_204[0][0]  │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_204      │ (None, 150, 150,  │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_205 (Conv2D) │ (None, 150, 150,  │     18,496 │ activation_204[0… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 150, 150,  │        256 │ conv2d_205[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_205      │ (None, 150, 150,  │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_206 (Conv2D) │ (None, 75, 75,    │     55,392 │ activation_205[0… │
│                     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 75, 75,    │        384 │ conv2d_206[0][0]  │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_4     │ (None, 75, 75,    │          0 │ activation_205[0… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_206      │ (None, 75, 75,    │          0 │ batch_normalizat… │
│ (Activation)        │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 75, 75,    │          0 │ max_pooling2d_4[… │
│ (Concatenate)       │ 160)              │            │ activation_206[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_210 (Conv2D) │ (None, 75, 75,    │      5,152 │ concatenate[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 75, 75,    │        128 │ conv2d_210[0][0]

 Total params: 13,050,705 (49.78 MB)

 Trainable params: 13,013,297 (49.64 MB)

 Non-trainable params: 37,408 (146.12 KB)

In [19]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

early_stop = EarlyStopping(
    patience=5,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    factor=0.3,
    patience=3,
    min_lr=1e-6
)

In [20]:
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=5,
    callbacks=[early_stop, reduce_lr]
)

Epoch 1/35
 280/1045 ━━━━━━━━━━━━━━━━━━━━ 22:48 2s/step - accuracy: 0.7871 - loss: 0.5514

KeyboardInterrupt: 